# 06 — Institutional Analytics

Quarterly portfolio summary (value, position count, top-5/top-10 concentration, **HHI**, average/median position, turnover) plus the league tables: largest holdings / buys / sells / new positions / full exits.

Turnover convention: `min(gross buys, gross sells) / average portfolio value` — counts round-trip trading only, so pure in/outflows don't inflate it.

In [1]:
# --- setup (run this cell first, every session) ---
import os, sys, pathlib

# Option A: Google Drive (data persists across sessions -- recommended)
# from google.colab import drive
# drive.mount("/content/drive")
# PROJECT_ROOT = pathlib.Path("/content/drive/MyDrive/13F-Analytics")

# Option B: uploaded zip directly to Colab (data lost on disconnect)
# PROJECT_ROOT = pathlib.Path("/content/13F-Analytics")

# Option C: running locally
PROJECT_ROOT = pathlib.Path(__file__).resolve().parents[1] if "__file__" in dir() else pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()

os.chdir(str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT))

# Install dependencies if needed (uncomment on Colab)
# import subprocess; subprocess.run(["pip", "install", "-q", "pyarrow", "requests", "pandas", "matplotlib", "numpy"])

import pandas as pd
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("Project root:", PROJECT_ROOT)
print("Ready.")

Project root: /content/13F-Analytics
Ready.


In [2]:
from src import analytics
from src.utils import load_parquet
from src import config

holdings = load_parquet(config.HOLDINGS_PARQUET)
tx = load_parquet(config.TRANSACTIONS_PARQUET)

summary = analytics.portfolio_summary(holdings, tx)
summary

,quarter,report_date,portfolio_value_usd,n_positions,top5_weight,top10_weight,hhi,avg_position_usd,median_position_usd,largest_holding,largest_holding_weight,gross_buys_usd,gross_sells_usd,turnover
0,2024Q2,2024-06-30,"279,969,062,343.00",41,0.73,0.90,0.15,"6,828,513,715.68","1,150,430,000.00",APPLE INC,0.30,NaN,NaN,NaN
1,2024Q3,2024-09-30,"266,378,900,503.00",40,0.71,0.90,0.13,"6,659,472,512.57","1,106,001,154.50",APPLE INC,0.26,"3,217,074,394.00","27,562,414,210.00",0.01
2,2024Q4,2024-12-31,"267,175,474,249.00",38,0.72,0.90,0.14,"7,030,933,532.87","1,143,632,802.50",APPLE INC,0.28,"2,248,887,301.00","5,326,180,888.00",0.01
3,2025Q1,2025-03-31,"1,106,550,356.00",4,1.00,1.00,0.46,"276,637,589.00","206,894,955.50",NUCOR CORP,0.63,"1,106,528,324.00","267,175,452,217.00",0.01
4,2025Q2,2025-06-30,"257,521,776,925.00",41,0.70,0.87,0.12,"6,281,018,949.39","1,186,820,921.00",APPLE INC,0.22,"256,416,002,972.00","776,403.00",0.00
5,2025Q3,2025-09-30,"267,334,501,955.00",41,0.70,0.87,0.12,"6,520,353,706.22","1,461,978,000.00",APPLE INC,0.23,"5,609,239,361.00","2,057,449,048.00",0.01
6,2025Q4,2025-12-31,"274,160,086,701.00",42,0.71,0.88,0.13,"6,527,621,111.93","1,292,417,438.50",APPLE INC,0.23,"4,800,187,902.00","5,616,032,717.00",0.02
7,2026Q1,2026-03-31,"263,095,703,570.00",29,0.67,0.91,0.12,"9,072,265,640.34","2,232,726,597.00",APPLE INC,0.22,"14,815,732,156.00","19,793,050,308.00",0.06


In [3]:
latest = summary["quarter"].iloc[-1]
print(f"=== {latest} ===")
analytics.largest_holdings(holdings, latest, n=10)

=== 2026Q1 ===


,issuer,asset_class,value_usd,portfolio_weight
0,APPLE INC,Common Stock,"57,843,260,493.00",0.22
1,AMERICAN EXPRESS CO,Common Stock,"45,859,204,536.00",0.17
2,COCA COLA CO,Common Stock,"30,420,000,000.00",0.12
3,BANK AMERICA CORP,Common Stock,"25,039,178,044.00",0.10
4,CHEVRON CORPORATION,Common Stock,"17,457,364,606.00",0.07
5,OCCIDENTAL PETE CORP,Common Stock,"17,221,193,015.00",0.07
6,ALPHABET INC,Common Stock,"15,600,071,913.00",0.06
7,CHUBB LTD SWITZ,Common Stock,"11,162,836,215.00",0.04
8,MOODYS CORP,Common Stock,"10,762,190,653.00",0.04
9,KRAFT HEINZ CO,Common Stock,"7,323,527,057.00",0.03


In [4]:
analytics.largest_buys(tx, latest, n=10)

,issuer,action,share_change,value_change_usd,value_usd_prev,value_usd_curr
0,ALPHABET INC,BUY,"36,403,656.00","10,014,229,467.00","5,585,842,446.00","15,600,071,913.00"
1,DELTA AIR LINES INC,NEW POSITION,"39,809,456.00","2,646,532,635.00",NaN,"2,646,532,635.00"
2,ALPHABET INC,NEW POSITION,"3,585,215.00","1,028,454,775.00",NaN,"1,028,454,775.00"
3,NEW YORK TIMES CO MTN BE,BUY,"10,080,791.00","916,555,428.00","351,663,948.00","1,268,219,376.00"
4,LENNAR CORP,BUY,"3,048,692.00","152,215,251.00","724,837,660.00","877,052,911.00"
5,MACYS INC,NEW POSITION,"3,038,355.00","54,963,842.00",NaN,"54,963,842.00"
6,LENNAR CORP,BUY,"56,723.00","2,780,758.00","17,214,818.00","19,995,576.00"


In [5]:
analytics.largest_sells(tx, latest, n=10)

,issuer,action,share_change,value_change_usd,value_usd_prev,value_usd_curr
0,BANK AMERICA CORP,SELL,"-3,671,769.00","-3,412,098,326.00","28,451,276,370.00","25,039,178,044.00"
1,VISA INC,FULL EXIT,"-8,297,460.00","-2,910,002,197.00","2,910,002,197.00",NaN
2,CHEVRON CORPORATION,SELL,"-45,780,506.00","-2,379,766,525.00","19,837,131,131.00","17,457,364,606.00"
3,MASTERCARD INCORPORATED,FULL EXIT,"-3,986,648.00","-2,275,897,610.00","2,275,897,610.00",NaN
4,CONSTELLATION BRANDS INC,SELL,"-12,367,110.00","-1,698,546,500.00","1,793,480,000.00","94,933,500.00"
5,UNITEDHEALTH GROUP INC,FULL EXIT,"-5,039,564.00","-1,663,610,472.00","1,663,610,472.00",NaN
6,DOMINOS PIZZA INC,FULL EXIT,"-3,350,000.00","-1,396,347,000.00","1,396,347,000.00",NaN
7,AON PLC,FULL EXIT,"-3,602,995.00","-1,271,424,876.00","1,271,424,876.00",NaN
8,POOL CORP,FULL EXIT,"-3,068,885.00","-702,007,444.00","702,007,444.00",NaN
9,AMAZON COM INC,FULL EXIT,"-2,276,000.00","-525,346,320.00","525,346,320.00",NaN


In [6]:
analytics.new_positions(tx, latest, n=10)

,issuer,action,share_change,value_change_usd,value_usd_prev,value_usd_curr
0,DELTA AIR LINES INC,NEW POSITION,"39,809,456.00","2,646,532,635.00",NaN,"2,646,532,635.00"
1,ALPHABET INC,NEW POSITION,"3,585,215.00","1,028,454,775.00",NaN,"1,028,454,775.00"
2,MACYS INC,NEW POSITION,"3,038,355.00","54,963,842.00",NaN,"54,963,842.00"


In [7]:
analytics.full_exits(tx, latest, n=10)

,issuer,action,share_change,value_change_usd,value_usd_prev,value_usd_curr
0,VISA INC,FULL EXIT,"-8,297,460.00","-2,910,002,197.00","2,910,002,197.00",NaN
1,MASTERCARD INCORPORATED,FULL EXIT,"-3,986,648.00","-2,275,897,610.00","2,275,897,610.00",NaN
2,UNITEDHEALTH GROUP INC,FULL EXIT,"-5,039,564.00","-1,663,610,472.00","1,663,610,472.00",NaN
3,DOMINOS PIZZA INC,FULL EXIT,"-3,350,000.00","-1,396,347,000.00","1,396,347,000.00",NaN
4,AON PLC,FULL EXIT,"-3,602,995.00","-1,271,424,876.00","1,271,424,876.00",NaN
5,POOL CORP,FULL EXIT,"-3,068,885.00","-702,007,444.00","702,007,444.00",NaN
6,AMAZON COM INC,FULL EXIT,"-2,276,000.00","-525,346,320.00","525,346,320.00",NaN
7,HEICO CORP NEW,FULL EXIT,"-1,294,612.00","-326,798,907.00","326,798,907.00",NaN
8,LIBERTY MEDIA CORP DEL,FULL EXIT,"-3,018,555.00","-297,357,853.00","297,357,853.00",NaN
9,CHARTER COMMUNICATIONS INC N,FULL EXIT,"-1,060,882.00","-221,459,118.00","221,459,118.00",NaN


In [8]:
analytics.asset_allocation(holdings, latest)

,asset_class,value_usd,weight
0,Common Stock,"263,095,703,570.00",1.00
